# 04 · Stacked hybrid: UBCF + CBF first, then add popularity

**Inputs:** the score files written by `01_ubcf`, `02_cbf`, `03_popularity`, plus user-state features.

**Day-block stacking (no leakage):**

| Phase | Base models built on | Scored day | Used for |
|---|---|---|---|
| `stack_train` | train days 1–4 | train day 5 | **training the stacker** (negative-sampled) |
| `val` | train days 1–5 | val day | early stopping / model choice / ablation |
| `test` | train days 1–5 | test day | final report, scored once |

Every score the stacker sees was produced by base models that had **not** seen that impression's click.

**Contract:** same split, full-slate evaluation with the shared tie-safe `evaluate()`, missing values → mean of the stacker's training set (+ availability flags), group negative-sampling policy on training rows only, no `slate_length` / `article_recency` / `session_dow`.

**Training order:** first fit stacks without popularity, then fit full stacks with popularity added.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import common as C
import importlib
importlib.reload(C)

beh = C.load_behaviors()
news = C.load_news()

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm not installed -> using sklearn HistGradientBoosting (pointwise) instead of LambdaRank")

## User-state features (per impression, information before t only)

In [2]:
def user_state_features(beh, news):
    b = beh[["imp_id", "user_id", "time", "history"]].sort_values(["user_id", "time"]).copy()
    b["log_hist_len"] = np.log1p(b["history"].str.len())
    # reading entropy over history categories (computed once per distinct history)
    cat = dict(zip(news["news_id"], news["category"]))
    b["_h"] = b["history"].str.join(" ")
    uniq = b[["_h"]].drop_duplicates()
    ex = uniq.assign(nid=uniq["_h"].str.split()).explode("nid").dropna()
    ex["cat"] = ex["nid"].map(cat)
    p = ex.groupby(["_h", "cat"]).size() / ex.groupby("_h").size()
    ent = (-(p * np.log(p))).groupby(level=0).sum()
    b["reading_entropy"] = b["_h"].map(ent)                      # NaN for empty history -> imputed
    # inactivity gap (hours since the user's previous impression); NaN on first impression
    b["inactivity_gap"] = b.groupby("user_id")["time"].diff().dt.total_seconds() / 3600
    b["first_imp"] = b["inactivity_gap"].isna().astype(np.int8)
    h = b["time"].dt.hour
    b["hour_sin"], b["hour_cos"] = np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)
    return b[["imp_id", "log_hist_len", "reading_entropy", "inactivity_gap", "first_imp", "hour_sin", "hour_cos"]]


ustate = user_state_features(beh, news)

## Assemble one frame per phase

In [3]:
KEYS = ["imp_id", "news_id"]


def load_phase(phase):
    u = C.load_scores("ubcf", phase).rename(columns={"score": "ubcf_score", "available": "ubcf_available"})
    c = C.load_scores("cbf", phase).rename(columns={"score": "cbf_score", "available": "cbf_available"})
    p = C.load_scores("pop", phase).rename(columns={"available": "pop_available"})
    base = ["user_id", "label", "user_cold", "item_cold"]
    for name, d in (("ubcf", u), ("cbf", c), ("pop", p)):
        if d.duplicated(KEYS).any():
            print(f"WARNING: duplicate (imp_id, news_id) in {name}/{phase} - keeping first")
    u, c, p = (d.drop_duplicates(KEYS) for d in (u, c, p))
    df = (u.merge(c.drop(columns=base), on=KEYS, how="inner")
           .merge(p.drop(columns=base + ["score"]), on=KEYS, how="inner")
           .merge(ustate, on="imp_id", how="left"))
    assert len(df) == len(u), f"{phase}: row mismatch after merge"
    return df


data = {ph: load_phase(ph) for ph in C.PHASES}
{ph: f"{len(d):,} rows / {d['imp_id'].nunique():,} impressions" for ph, d in data.items()}

{'stack_train': '1,231,673 rows / 31,624 impressions',
 'val': '1,222,429 rows / 30,270 impressions',
 'test': '2,740,998 rows / 73,152 impressions'}

## Features
Group **A** collaborative, **B** content, **C** popularity, **D** user state. Each imputed signal has its availability flag next to it so "unknown" ≠ "average".

`pop_log_exps` (prior exposure count) is **off by default**: it partly acts as a recency proxy, and `article_recency` is banned by the contract – agree with the group before switching it on.

In [4]:
USE_EXPOSURE_COUNT = False

GROUPS = {
    "A_ubcf": ["ubcf_score", "ubcf_available", "ubcf_strength", "ubcf_n_clicks"],
    "B_cbf": ["cbf_content", "cbf_kg", "cbf_cat_aff", "cbf_available"],
    "C_pop": ["pop_ctr", "pop_available", "item_cold"] + (["pop_log_exps"] if USE_EXPOSURE_COUNT else []),
    "D_user": ["user_cold", "log_hist_len", "reading_entropy", "hour_sin", "hour_cos"],
}
FEATURES = [f for g in GROUPS.values() for f in g]

# contract: missing values -> mean of the (stacker's) training set
means = C.impute_train_mean(data["stack_train"], list(data.values()), FEATURES)
train = C.sample_negatives(data["stack_train"], k=C.NEG_K)
print(f"stacker training rows after negative sampling: {len(train):,} "
      f"(positives {train['label'].mean():.1%})")

stacker training rows after negative sampling: 230,137 (positives 20.8%)


## Shared stacker functions
Define LR, GBM, and prediction helpers before training either feature set. LightGBM uses LambdaRank with validation early stopping; otherwise GBM uses the scikit-learn classifier fallback.

In [5]:
def fit_lr(feats):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    m.fit(train[feats], train["label"])
    return m

def fit_gbm(feats):
    if HAS_LGB:
        tr = train.sort_values("imp_id")
        va = data["val"].sort_values("imp_id")
        m = lgb.LGBMRanker(objective="lambdarank", n_estimators=2000, learning_rate=0.05,
                           num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=C.SEED, verbose=-1)
        m.fit(tr[feats], tr["label"], group=tr.groupby("imp_id", sort=False).size().to_numpy(),
              eval_set=[(va[feats], va["label"])],
              eval_group=[va.groupby("imp_id", sort=False).size().to_numpy()],
              eval_at=[5, 10], callbacks=[lgb.early_stopping(100, verbose=False)])
        return m
    from sklearn.ensemble import HistGradientBoostingClassifier
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True,
                                       random_state=C.SEED)
    return m.fit(train[feats], train["label"])

def predict(m, df, feats):
    if HAS_LGB and isinstance(m, lgb.LGBMRanker):
        return m.predict(df[feats])
    return m.predict_proba(df[feats])[:, 1]


## Stage 1 · Train without popularity
Train LR and GBM using UBCF, CBF, and user-state features. Exclude the entire `C_pop` group. Both stages use the same sampled training rows and evaluation splits.

In [6]:
# Keep collaborative, content, and user-state features; exclude the entire popularity group.
FEATURES_NO_POP = [f for group, feats in GROUPS.items() if group != "C_pop" for f in feats]
lr_no_pop = fit_lr(FEATURES_NO_POP)
gbm_no_pop = fit_gbm(FEATURES_NO_POP)

no_pop_results = {}
for name, model, score_name in (
    ("Hybrid-LR-no-pop", lr_no_pop, "hybrid_lr_no_pop"),
    ("Hybrid-GBM-no-pop", gbm_no_pop, "hybrid_gbm_no_pop"),
):
    for phase in ("val", "test"):
        scored = data[phase].assign(score=predict(model, data[phase], FEATURES_NO_POP), available=1)
        C.save_scores(scored, score_name, phase)
        metrics = C.evaluate(scored)[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
        no_pop_results[(name, phase)] = metrics
        metrics.to_csv(f"{C.OUT_DIR}/metrics_{score_name}_{phase}.csv")
        print(f"\n=== {name} · {phase} ===")
        print(metrics.round(4).to_string())


/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



=== Hybrid-LR-no-pop · val ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          30270  0.3482  0.3246   0.3819  0.6429
Cold users (0 history)                         772  0.2902  0.2733   0.3254  0.5809
Sparse users (1-4 history)                    3247  0.3238  0.3147   0.3752  0.6005
Warm users (>=5 history)                     26251  0.3529  0.3274   0.3844  0.6500
Clicked article seen before                   8276  0.2331  0.2128   0.2645  0.5517
Clicked article new (no prior clicks)        21994  0.3915  0.3667   0.4260  0.6773

=== Hybrid-LR-no-pop · test ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          73152  0.3623  0.3462   0.4048  0.6433
Cold users (0 history)                        2214  0.2638  0.2436   0.3020  0.5095
Sparse users (1-4 history)                    8092  0.3550  0.3564   0.4118  0.5954
Warm users 

## Stage 2 · Add popularity and train full stacks
Add the `C_pop` feature group and fit fresh LR and GBM models for comparison. These are separate fits, rather than continued training of the stage 1 models.

In [7]:
lr = fit_lr(FEATURES)

coef = pd.Series(lr[-1].coef_[0], index=FEATURES).sort_values(key=np.abs, ascending=False)

print("LR standardised coefficients:\n", coef.round(3).to_string())

gbm = fit_gbm(FEATURES)

if HAS_LGB:
    imp = pd.Series(gbm.booster_.feature_importance("gain"), index=FEATURES)
    print("best iteration:", gbm.best_iteration_)
    print("gain importance:\n", (imp / imp.sum()).sort_values(ascending=False).round(3).to_string())


LR standardised coefficients:
 cbf_content        0.448
log_hist_len      -0.208
ubcf_n_clicks      0.164
cbf_cat_aff        0.161
pop_ctr            0.145
ubcf_available    -0.108
item_cold          0.085
ubcf_score        -0.069
cbf_kg             0.067
ubcf_strength      0.053
reading_entropy    0.052
pop_available      0.039
user_cold         -0.020
cbf_available      0.020
hour_cos          -0.019
hour_sin           0.019


/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


best iteration: 79
gain importance:
 cbf_content        0.376
pop_ctr            0.224
cbf_cat_aff        0.128
cbf_kg             0.087
log_hist_len       0.032
ubcf_score         0.031
hour_sin           0.028
ubcf_n_clicks      0.022
ubcf_available     0.020
hour_cos           0.019
ubcf_strength      0.010
reading_entropy    0.009
item_cold          0.009
pop_available      0.006
user_cold          0.000
cbf_available      0.000


## Ablation on validation – what does each feature group add?

In [8]:
abl = []
for drop in [None] + list(GROUPS):
    feats = [f for f in FEATURES if drop is None or f not in GROUPS[drop]]
    m = fit_gbm(feats)
    r = C.evaluate(data["val"].assign(score=predict(m, data["val"], feats)))
    abl.append({"dropped": drop or "(none)",
                **{f"{seg} nDCG@10": r.loc[seg, "nDCG@10"] for seg in r.index}})
ablation = pd.DataFrame(abl).set_index("dropped")
ablation.round(4)

/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/charlenephua/.venvs/bt4222/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.


,All nDCG@10,Cold users (0 history) nDCG@10,Sparse users (1-4 history) nDCG@10,Warm users (>=5 history) nDCG@10,Clicked article seen before nDCG@10,Clicked article new (no prior clicks) nDCG@10
dropped,,,,,,
(none),0.3950,0.3409,0.3945,0.3966,0.3487,0.4123
A_ubcf,0.3941,0.3408,0.3953,0.3955,0.3518,0.4100
B_cbf,0.3045,0.3027,0.3301,0.3014,0.3565,0.2849
C_pop,0.3833,0.3259,0.3781,0.3856,0.3044,0.4129
D_user,0.3955,0.3391,0.3928,0.3974,0.3417,0.4157


## Final comparison: base models vs stacked hybrid (val & test, by segment)

In [9]:
results = dict(no_pop_results)
for model in ("ubcf", "cbf", "pop"):
    for ph, t in C.standalone_eval(model).items():
        results[(model.upper(), ph)] = t
for name, m in (("Hybrid-LR", lr), ("Hybrid-GBM", gbm)):
    for ph in ("val", "test"):
        d = data[ph].assign(score=predict(m, data[ph], FEATURES))
        results[(name, ph)] = C.evaluate(d)
        if name == "Hybrid-GBM":
            d["available"] = 1
            C.save_scores(d, "hybrid", ph)

summary = pd.concat({k: v for k, v in results.items()}, names=["model", "phase", "segment"])
summary = summary[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
summary.to_csv(f"{C.OUT_DIR}/metrics_all_models.csv")

ORDER = ["UBCF", "CBF", "POP", "Hybrid-LR-no-pop", "Hybrid-GBM-no-pop", "Hybrid-LR", "Hybrid-GBM"]
SEGS = list(results[("UBCF", "val")].index)
for ph in ("val", "test"):
    print(f"\n===== {ph.upper()} =====")
    s = summary.xs(ph, level="phase")
    print("\nAll impressions:")
    print(s.xs("All", level="segment").reindex(ORDER).round(4).to_string())
    print("\nnDCG@10 by segment:")
    print(s["nDCG@10"].unstack("segment").reindex(index=ORDER, columns=SEGS).round(4).to_string())


===== VAL =====

All impressions:
                   impressions     MRR  nDCG@5  nDCG@10     AUC
model                                                          
UBCF                     30270  0.2546  0.2276   0.2866  0.5304
CBF                      30270  0.3448  0.3207   0.3798  0.6428
POP                      30270  0.2737  0.2436   0.3060  0.5723
Hybrid-LR-no-pop         30270  0.3482  0.3246   0.3819  0.6429
Hybrid-GBM-no-pop        30270  0.3507  0.3254   0.3833  0.6434
Hybrid-LR                30270  0.3535  0.3301   0.3894  0.6567
Hybrid-GBM               30270  0.3608  0.3355   0.3950  0.6629

nDCG@10 by segment:
segment               All  Cold users (0 history)  Sparse users (1-4 history)  Warm users (>=5 history)  Clicked article seen before  Clicked article new (no prior clicks)
model                                                                                                                                                                      
UBCF               0.286

## Performance by user group
Groups use the length of the impression's raw `history`, the same definition as the MLP slices: **cold** = 0, **sparse** = 1–4, **warm** = ≥5. Every model is ranked on the same impressions, on full slates.

In [10]:
USER_GROUPS = ["All", *C.USER_SEGS]
by_user = summary[summary.index.get_level_values("segment").isin(USER_GROUPS)]
by_user.to_csv(f"{C.OUT_DIR}/metrics_user_groups.csv")

for ph in ("val", "test"):
    print(f"\n===== {ph.upper()} =====")
    for seg in USER_GROUPS:
        s = by_user.xs((ph, seg), level=("phase", "segment")).reindex(ORDER)
        print(f"\n{seg} - {int(s['impressions'].iloc[0]):,} impressions")
        print(s.drop(columns="impressions").round(4).to_string())


===== VAL =====

All - 30,270 impressions
                      MRR  nDCG@5  nDCG@10     AUC
model                                             
UBCF               0.2546  0.2276   0.2866  0.5304
CBF                0.3448  0.3207   0.3798  0.6428
POP                0.2737  0.2436   0.3060  0.5723
Hybrid-LR-no-pop   0.3482  0.3246   0.3819  0.6429
Hybrid-GBM-no-pop  0.3507  0.3254   0.3833  0.6434
Hybrid-LR          0.3535  0.3301   0.3894  0.6567
Hybrid-GBM         0.3608  0.3355   0.3950  0.6629

Cold users (0 history) - 772 impressions
                      MRR  nDCG@5  nDCG@10     AUC
model                                             
UBCF               0.2572  0.2347   0.2841  0.5261
CBF                0.2851  0.2669   0.3233  0.5814
POP                0.2705  0.2420   0.3064  0.5784
Hybrid-LR-no-pop   0.2902  0.2733   0.3254  0.5809
Hybrid-GBM-no-pop  0.2858  0.2668   0.3259  0.5777
Hybrid-LR          0.2988  0.2811   0.3367  0.6000
Hybrid-GBM         0.3045  0.2819   0.3409  0.61